# Modules Built Around Core

MolPy is designed with a modular architecture where the `core` module provides the fundamental data structures (`Frame`, `Block`, `Box`, `Atomistic`), and all other modules are built on top of it. This guide introduces the key modules that extend MolPy's functionality.

## Overview

The modules covered in this guide:

- **Adapter**: Bridge to external libraries (RDKit)
- **Compute**: Unified computation abstraction
- **Engine**: Interfaces to simulation engines (LAMMPS, CP2K)
- **IO**: File format readers and writers
- **Op**: Geometric operations
- **Pack**: Molecular packing tools
- **Parser**: SMILES/SMARTS parsing
- **Potential**: Force field potential functions
- **Reacter**: Chemical reaction modeling
- **Typifier**: Automatic atom typing


## Adapter Module

The `adapter` module provides bidirectional conversion between MolPy structures and external molecular libraries, primarily RDKit.

### Key Features

- Convert between RDKit `Mol` objects and MolPy `Atomistic` structures
- Generate 3D coordinates using RDKit's ETKDG method
- Optimize geometries with MMFF/UFF force fields
- Generate 2D molecular drawings (SVG)
- Stable atom mapping for round-trip conversion


In [ ]:
from molpy.adapter import RDKitWrapper, atomistic_to_mol, mol_to_atomistic
from molpy.core.atomistic import Atomistic
from rdkit import Chem

# Create a molecule from SMILES
mol = Chem.MolFromSmiles("CCO")

# Convert to MolPy Atomistic
atomistic = mol_to_atomistic(mol)
print(f"Converted molecule with {len(atomistic.atoms)} atoms")

# Create wrapper for bidirectional conversion
wrapper = RDKitWrapper.from_mol(mol)

# Generate 3D coordinates
wrapper.generate_3d(optimize=True)

# Access the MolPy structure
atomistic_3d = wrapper.inner
print(f"Generated 3D structure with coordinates")


### Converting from Atomistic to RDKit

You can also convert MolPy structures to RDKit:


In [ ]:
# Convert Atomistic back to RDKit
mol_from_atomistic = atomistic_to_mol(atomistic_3d)
print(f"RDKit molecule has {mol_from_atomistic.GetNumAtoms()} atoms")

# Generate 2D drawing
svg = wrapper.draw(show=False)
print("Generated 2D molecular drawing")


## Compute Module

The `compute` module provides a unified abstraction for defining and executing computational operations on molecular structures.

### Core Concepts

- **Result**: Base class for computation outputs
- **Compute**: Abstract base class for computation operations
- **ComputeContext**: Optional context for sharing expensive intermediates


In [ ]:
from molpy.compute import Compute, Result, ComputeContext
from molpy.core.frame import Frame
from dataclasses import dataclass

# Define a custom result
@dataclass
class RadiusOfGyrationResult(Result):
    """Result from radius of gyration calculation."""
    rg: float = 0.0

# Define a custom compute
class RadiusOfGyrationCompute(Compute[Frame, RadiusOfGyrationResult]):
    """Compute radius of gyration for a frame."""
    
    def compute(self, input: Frame) -> RadiusOfGyrationResult:
        if "atoms" not in input:
            return RadiusOfGyrationResult(name="radius_of_gyration", rg=0.0)
        
        atoms = input["atoms"]
        import numpy as np
        
        # Extract coordinates
        x = atoms["x"][:]
        y = atoms["y"][:]
        z = atoms["z"][:]
        coords = np.column_stack([x, y, z])
        
        # Calculate center of mass
        com = np.mean(coords, axis=0)
        
        # Calculate radius of gyration
        r_squared = np.sum((coords - com) ** 2, axis=1)
        rg = np.sqrt(np.mean(r_squared))
        
        return RadiusOfGyrationResult(
            name="radius_of_gyration",
            rg=float(rg)
        )

# Use the compute
compute = RadiusOfGyrationCompute()
# result = compute(frame)  # Uncomment when you have a frame


### Sharing Context Between Computes

You can share expensive intermediate computations using `ComputeContext`:


In [ ]:
# Create shared context
context = ComputeContext()
# context.data["neighbor_list"] = compute_neighbor_list(frame)

# Use in multiple computes
# compute1 = SomeCompute(context=context)
# compute2 = AnotherCompute(context=context)
# Both can access the neighbor list from context


## Engine Module

The `engine` module provides interfaces to external simulation engines like LAMMPS and CP2K.

### Supported Engines

- **LAMMPSEngine**: Interface to LAMMPS molecular dynamics simulator
- **CP2KEngine**: Interface to CP2K quantum chemistry package


In [ ]:
from molpy.engine import LAMMPSEngine, CP2KEngine
from molpy.core.script import Script

# Create LAMMPS input script
script = Script.from_text(
    name="input",
    text="""units real
atom_style full
read_data system.data
pair_style lj/cut/coul/long 10.0
pair_coeff * * 0.0 0.0
run 1000
""",
    language="other"
)

# Create engine and prepare
engine = LAMMPSEngine(executable="lmp")
# engine.prepare(work_dir="./calc", scripts=script)

# Run calculation
# result = engine.run()
# print(f"Return code: {result.returncode}")


## IO Module

The `io` module provides comprehensive file format support for reading and writing molecular structures and trajectories.

### Structure Formats

- **PDB**: Protein Data Bank format
- **XYZ**: Simple coordinate format
- **LAMMPS DATA**: LAMMPS data file format
- **GRO**: GROMACS structure format
- **MOL2**: Tripos MOL2 format
- **XSF**: XCrySDen structure format


In [ ]:
from molpy.io import read_pdb, write_pdb, read_xyz, read_lammps_data

# Read a PDB file
# frame = read_pdb("structure.pdb")

# Write to XYZ format
# write_xyz("structure.xyz", frame)

# Read LAMMPS data file
# frame = read_lammps_data("system.data", atom_style="full")

print("IO module supports multiple file formats")


### Trajectory Formats

The IO module also supports trajectory formats:


In [ ]:
from molpy.io import read_lammps_trajectory, read_xyz_trajectory
from molpy.core.trajectory import Trajectory

# Read trajectory
# traj = read_lammps_trajectory("trajectory.dump")

# Access frames
# frame_0 = traj[0]
# frame_100 = traj[100]


## Op Module

The `op` module provides geometric operations for manipulating molecular coordinates.

### Available Operations

- **Rotation**: Rodrigues rotation, quaternion rotation
- **Translation**: Coordinate translation
- **Matrix operations**: Rotation matrix from vectors


In [ ]:
import numpy as np
from molpy.op.geometry import (
    rotate_by_rodrigues,
    rotate_by_quaternion,
    translate,
    rotation_matrix_from_vectors
)

# Rotate coordinates using Rodrigues formula
coords = np.array([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0]])
axis = np.array([0.0, 0.0, 1.0])
theta = np.pi / 2  # 90 degrees

rotated = rotate_by_rodrigues(coords, axis, theta)
print(f"Rotated coordinates:\n{rotated}")

# Translate coordinates
vector = np.array([1.0, 1.0, 1.0])
translated = translate(coords, vector)
print(f"Translated coordinates:\n{translated}")


## Pack Module

The `pack` module provides tools for packing molecules into simulation boxes, typically using external tools like Packmol.

### Key Components

- **Molpack**: Main packing interface
- **Target**: Define packing targets and constraints
- **Packer**: Backend packers (Packmol, etc.)


In [ ]:
from molpy.pack import Molpack, Target, get_packer
from molpy.core.box import Box
import numpy as np

# Create a packing target
box = Box.from_lengths_angles([20.0, 20.0, 20.0], [90.0, 90.0, 90.0])

# Define packing constraints
# target = Target(
#     structure=atomistic,
#     count=100,
#     box=box
# )

# Create packer
# packer = get_packer("packmol")

# Pack molecules
# molpack = Molpack(packer=packer)
# packed = molpack.pack([target])

print("Pack module provides molecular packing capabilities")


## Parser Module

The `parser` module provides SMILES and SMARTS parsing capabilities.

### Parsers

- **SmilesParser**: Parse SMILES strings
- **SmartsParser**: Parse SMARTS patterns


In [ ]:
from molpy.parser import SmilesParser, SmartsParser

# Parse SMILES string
smiles_parser = SmilesParser()
smiles_ir = smiles_parser.parse_smiles("CCO")
print(f"Parsed SMILES: {smiles_ir}")

# Parse SMARTS pattern
smarts_parser = SmartsParser()
smarts_ir = smarts_parser.parse_smarts("[C;H2,H3]")  # Carbon with 2 or 3 hydrogens
print(f"Parsed SMARTS: {smarts_ir}")


## Potential Module

The `potential` module provides force field potential functions for bonds, angles, dihedrals, and pair interactions.

### Potential Types

- **Bond potentials**: Harmonic bonds
- **Angle potentials**: Harmonic angles
- **Pair potentials**: Lennard-Jones, Coulombic


In [ ]:
from molpy.potential import Potential, Potentials

# The potential module provides force field potential functions
# for bonds, angles, dihedrals, and pair interactions.
# 
# Example usage:
# from molpy.potential.bond import HarmonicBond
# from molpy.potential.angle import HarmonicAngle
# from molpy.potential.pair import LennardJones, Coulombic
#
# bond_pot = HarmonicBond(k=100.0, r0=1.5)
# angle_pot = HarmonicAngle(k=50.0, theta0=109.47)
# lj_pot = LennardJones(epsilon=0.1, sigma=3.4)
# coul_pot = Coulombic()

print("Potential module provides force field functions")


## Reacter Module

The `reacter` module provides a framework for defining and executing chemical reactions within MolPy.

### Key Concepts

- **Reacter**: Represents a single chemical reaction type
- **ProductSet**: Container for reaction products and metadata
- **Selectors**: Functions that identify anchor atoms and leaving groups
- **Transformers**: Functions that create or modify bonds


In [ ]:
from molpy.reacter import (
    Reacter,
    port_anchor_selector,
    remove_one_H,
    make_single_bond
)

# Define a C-C coupling reaction
cc_coupling = Reacter(
    name="C-C_coupling_with_H_loss",
    anchor_left=port_anchor_selector,
    anchor_right=port_anchor_selector,
    leaving_left=remove_one_H,
    leaving_right=remove_one_H,
    bond_maker=make_single_bond,
)

# Execute reaction between two monomers
# product = cc_coupling.run(monoA, monoB, port_L="1", port_R="2")
# print(f"Removed atoms: {product.notes['removed_atoms']}")
# print(f"New bonds: {product.notes['new_bonds']}")

print("Reacter module enables chemical reaction modeling")


## Typifier Module

The `typifier` module provides automatic atom typing for force field assignment.

### Key Components

- **LayeredTypingEngine**: Multi-layer typing system
- **OplsAtomisticTypifier**: OPLS-AA force field typifier
- **DependencyAnalyzer**: Analyzes typing dependencies


In [ ]:
from molpy.typifier import OplsAtomisticTypifier, LayeredTypingEngine

# Create typifier
# typifier = OplsAtomisticTypifier()

# Assign types to structure
# typed_structure = typifier.assign_types(atomistic)

# Use layered engine for complex typing
# engine = LayeredTypingEngine()
# typed = engine.type(atomistic)

print("Typifier module provides automatic atom typing")


## Summary

All modules in MolPy are designed to work with the core data structures:

- **Frame** and **Block**: Used by IO, Compute, Engine modules
- **Atomistic**: Used by Adapter, Reacter, Typifier, Builder modules
- **Box**: Used by Pack, Engine, IO modules

This unified data model ensures that all modules can work together seamlessly, making it easy to build complex molecular simulation workflows.
